# 06.01_Auco_DEGs_GOs_analysis_R

Auco 细胞簇差异基因与 GO 功能富集。

- 当前文件：`analysis/06_single_cell_analysis/06.01_Auco_DEGs_GOs_analysis_R.ipynb`
- 原始来源：`Codes/06.01_R_GOs_analysis_Auco.ipynb`（旧编号仅用于溯源）。
- 运行内核：**R**。
- 导入依赖：`Seurat`, `SeuratObject`, `clusterProfiler`, `data.table`, `dplyr`, `enrichplot`, `ggVennDiagram`, `ggplot2`, `ontologyIndex`, `stringr`, `tidyverse`。
- 当前编号与流程见 `docs/workflow.md`、`docs/code_index.md`。
- 仅更新整理版导读；原分析单元格、参数和顺序保持不变。原始 cell 索引在本文件中加 1。


r_base

In [ ]:
packageVersion('clusterProfiler')

In [ ]:
fig_dir <- '/share/home/zhangze/zz/NeuralOrigin/Figures'

## 水母单细胞自聚类 GO 富集分析

### 1.引入必要的包

In [ ]:
library(data.table)
library(stringr)
library(dplyr)
library(Seurat)
library(ggplot2)
library(clusterProfiler)

### 2.读入eggNOG数据提取GOs

In [ ]:
# 读取eggNOG注释文件，tab分隔，有表头
egg<-as.data.frame(fread("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/eggNOG_output_processed/Auco.emapper.annotations.tsv"))
head(egg)

# 查看数据维度和前几行，确认读取成功
dim(egg)

# 把空字符串替换为NA，方便后续过滤
egg[egg == ""] <- NA

# 只保留query和GOs列，去除含NA的行（没有GO注释的基因）
gterms <- egg %>%
  select(query, GOs) %>%
  na.omit()

# gene_ids 是所有基因ID
gene_ids <- egg$query

# 找出所有含有GO注释的行的逻辑向量
eggnog_lines_with_go <- egg$GOs != ""

# 对含GO注释的行，按逗号拆分GO字符串，得到列表，每个元素是一个GO ID向量
eggnog_annotations_go <- str_split(egg[eggnog_lines_with_go, ]$GOs, ",")

# 构造gene-to-GO的长格式数据框：
# gene列重复对应基因ID若干次（对应每个GO ID的数量）
# term列为拆分的GO ID，展平成一列
gene_to_go <- data.frame(
  gene = rep(gene_ids[eggnog_lines_with_go], times = sapply(eggnog_annotations_go, length)),
  term = unlist(eggnog_annotations_go)
)

# 查看前几行结果
head(gene_to_go)
dim(gene_to_go)

### 3.读入对应的 Seurat 数据

In [ ]:
# 读取Seurat对象
seurat_obj <- readRDS("/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/scMatrix/AU_scMatrix_CycloneSeq/analysis_results/Auco.seurat.rds")
# seurat_obj <- readRDS("/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/scMatrix/AU_scMatrix/analysis_results/Auco.seurat.rds")
seurat_obj

In [ ]:
# 绘制UMAP图，默认用seurat_obj@meta.data$seurat_clusters作为cluster标签
DimPlot(seurat_obj, reduction = "umap", label = TRUE) + ggtitle("UMAP of Auco clusters")

### 4.筛选出每个类的差异基因

In [ ]:
# 找每个cluster的top DEGs（默认方法：FindAllMarkers）
# 这里举例用Wilcox检验，筛选表达在至少10%细胞且logFC>0.25的基因
markers <- FindAllMarkers(seurat_obj, 
                          only.pos = TRUE, 
                          min.pct = 0.3, 
                          logfc.threshold = 0.95,
                          test.use = "wilcox")
head(markers)

In [ ]:
# 按cluster分组，并组内按avg_log2FC降序排序
markers <- markers %>%
  group_by(cluster) %>%
  arrange(desc(avg_log2FC), .by_group = TRUE) %>%
  ungroup()

In [ ]:
# 统计每个cluster的marker基因数量
marker_counts <- markers %>%
  group_by(cluster) %>%
  summarise(marker_num = n())

marker_counts

In [ ]:
# 提取每个cluster排名前5的marker基因
top5_markers <- markers %>%
  group_by(cluster) %>%
  slice_head(n = 5)

print(top5_markers)

In [ ]:
# 绘制前5 marker基因的热图
DoHeatmap(seurat_obj, features = top5_markers$gene) + ggtitle("Top 5 markers heatmap")

### 5.使用 ClusterProfiler 进行富集

In [ ]:
# 选取cluster 0的marker基因列表
cluster_id <- "7"  # 注意cluster列是字符还是数字，要对应
gene_list <- markers %>% 
  filter(cluster == cluster_id) %>% 
  pull(gene)

length(gene_list)
print(gene_list)

# 准备term2gene，列名和顺序必须是term, gene
term2gene <- gene_to_go[, c("term", "gene")]
sum(gene_list %in% term2gene$gene)

# 富集分析
ego <- enricher(gene = gene_list,
                TERM2GENE = term2gene,
                pvalueCutoff = 0.3,
                pAdjustMethod = "BH")

# 查看结果
# head(ego)
dim(ego)

# 画图
# barplot(ego, showCategory = 10) + ggtitle(paste0("GO enrichment for cluster ", cluster_id))
# dotplot(ego, showCategory = 10) + ggtitle(paste0("GO enrichment for cluster ", cluster_id))

# 转成数据框方便后续处理，保存
df <- as.data.frame(ego)
write.csv(df, file = paste0("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/cluster_GO_results/cluster", cluster_id, "_GO_enrich.csv"), row.names = FALSE)

# 使用go2term获取Term表，并左连接合并
df1 <- go2term(df$ID)
df <- left_join(df, df1, by = c("ID" = "go_id"))
df$term <- df$Term
df$Term <- NULL

# 使用go2ont获取Ontology表，并左连接合并
df2 <- go2ont(df$ID)
df <- left_join(df, df2, by = c("ID" = "go_id"))
df$Ont <- df$Ontology
df$Ontology <- NULL

# 4. 选择需要的列
# df3 <- df %>% select(term, Ont, pvalue)
df3 <- df %>%
  select(term, Ont, pvalue) %>%
  filter(!is.na(term) & !is.na(Ont))  # 去除term或Ont为NA的行

# 只取前10个显著GO term（按p值升序）
df3_top <- df3 %>%
  filter(pvalue < 0.05) %>%
  arrange(pvalue) %>%
  slice_head(n = 10)

# 5. 绘图保存：GO term的-log10(pvalue)柱状图，按Ontology分类着色，并按pvalue排序
p_barplot <- ggplot(df3_top, aes(x = reorder(term, -log10(pvalue)), y = -log10(pvalue))) +
  geom_col(aes(fill = Ont)) +
  coord_flip() +  # 横向显示标签
  labs(x = "", title = paste0("GO enrichment for cluster ", cluster_id)) +
  theme_bw()

pdf(file = paste0("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/cluster_GO_results/cluster", cluster_id, "_go_barplot.pdf"), width = 8, height = 5)
print(p_barplot)
dev.off()

# 6. 绘图保存：GO term的-dotplot，按Ontology分类着色，并按pvalue排序
p_dotplot <-ggplot(df3_top, aes(x = reorder(term, -log10(pvalue)), y = -log10(pvalue))) +
  geom_point(aes(size = -log10(pvalue), color = Ont)) +  # 使用点表示
  scale_size_continuous(range = c(3, 10)) +  # 控制点的大小范围
  coord_flip() +  # 横向显示标签
  labs(x = "", title = paste0("GO enrichment for cluster ", cluster_id)) +
  theme_bw()
  theme(legend.position = "bottom")  # 设置图例位置

pdf(file = paste0("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/cluster_GO_results/cluster", cluster_id, "_go_dotplot.pdf"), width = 8, height = 5)
print(p_dotplot)
dev.off()

# 7. 并排展示图片
p_barplot
p_dotplot

In [ ]:
# 识别类别总数
all_clusters <- unique(markers$cluster)

# 定义输出路径
save_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/cluster_GO_results/"

for(cluster_id in all_clusters) {
  cat("Processing cluster", cluster_id, "...\n")
  
  # 1. 提取当前cluster的marker基因
  gene_list <- markers %>%
    filter(cluster == cluster_id) %>%
    pull(gene)
  
  # 2. 富集分析
  ego <- enricher(
    gene = gene_list,
    TERM2GENE = gene_to_go[, c("term", "gene")],
    pvalueCutoff = 0.3,
    pAdjustMethod = "BH"
  )
  
  if (is.null(ego) || nrow(as.data.frame(ego)) == 0) {
    cat("No enrichment results for cluster", cluster_id, "\n")
    next
  }
  
  # 3. 转数据框并保存
  df <- as.data.frame(ego)
  write.csv(df, file = paste0(save_dir, "cluster", cluster_id, "_GO_enrich.csv"), row.names = FALSE)
  
  # 4. 注释GO term和Ontology
  df1 <- go2term(df$ID)
  df <- left_join(df, df1, by = c("ID" = "go_id"))
  df$term <- df$Term
  df$Term <- NULL
  df2 <- go2ont(df$ID)
  df <- left_join(df, df2, by = c("ID" = "go_id"))
  df$Ont <- df$Ontology
  df$Ontology <- NULL
  
  # 5. 数据筛选
  df3 <- df %>%
    select(term, Ont, pvalue) %>%
    filter(!is.na(term) & !is.na(Ont))
  df3_top <- df3 %>%
    filter(pvalue < 0.05) %>%
    arrange(pvalue) %>%
    slice_head(n = 10)
  
  if(nrow(df3_top) == 0){
    cat("No significant GO terms for cluster", cluster_id, "\n")
    next
  }
  
  # 6. 绘图
  p_barplot <- ggplot(df3_top, aes(x = reorder(term, -log10(pvalue)), y = -log10(pvalue))) +
    geom_col(aes(fill = Ont)) +
    coord_flip() +
    labs(x = "", title = paste0("GO enrichment for cluster ", cluster_id)) +
    theme_bw()
  
  pdf(file = paste0(save_dir, "cluster", cluster_id, "_go_barplot.pdf"), width = 8, height = 5)
  print(p_barplot)
  dev.off()
  
  p_dotplot <- ggplot(df3_top, aes(x = reorder(term, -log10(pvalue)), y = -log10(pvalue))) +
    geom_point(aes(size = -log10(pvalue), color = Ont)) +
    scale_size_continuous(range = c(3, 10)) +
    coord_flip() +
    labs(x = "", title = paste0("GO enrichment for cluster ", cluster_id)) +
    theme_bw() +
    theme(legend.position = "bottom")
  
  pdf(file = paste0(save_dir, "cluster", cluster_id, "_go_dotplot.pdf"), width = 8, height = 5)
  print(p_dotplot)
  dev.off()
}

# 非模式物种基因富集分析

### 0.Set up the environment

In [ ]:
library(data.table)
library(stringr)
library(dplyr)
library(Seurat)
library(ggplot2)
library(clusterProfiler)
library(tidyverse)
library(ontologyIndex)

### 1.Preparing Term to Gene table

In [ ]:
# prepare the term to gene table
eggNOG <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/eggNOG_output_processed/Auco.emapper.annotations.tsv") %>%
    dplyr::select(GOs, `query`) %>%
    dplyr::filter(GOs != "-") %>%
    separate_rows(GOs, sep = ",") %>%
    # mutate(gene = gsub("\\..*", "", `query`)) %>%
    select(GOs, gene = query) %>%
    distinct() %>%
    drop_na()
colnames(eggNOG) <- c("term", "gene")

In [ ]:
head(eggNOG)

### 2.Preparing Term to name table

In [ ]:
# prepare the term to name table
ontology <- get_ontology(file = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/go.obo",
                         propagate_relationships = "is_a",
                         extract_tags = "everything",
                         merge_equivalent_terms = TRUE)
eggNOG_term <- eggNOG %>%
    mutate(name = ontology$name[term]) %>%
    select(c(term, name)) %>%
    distinct() %>%
    drop_na() %>%
    filter(!grepl("obsolete", name))

eggNOG <- eggNOG %>%
    filter(term %in% eggNOG_term$term)

In [ ]:
write_tsv(x = eggNOG, file = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/term2gene_GO.tsv")
write_tsv(x = eggNOG_term, file = "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/term2name_GO.tsv")

### 3.Background gene list

In [ ]:
# 读取Seurat对象
seurat_obj <- readRDS("/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/scMatrix/AU_scMatrix_CycloneSeq/analysis_results/Auco.seurat.rds")
# seurat_obj <- readRDS("/share/home/zhangze/zz/NeuralOrigin/Data/03.SingleCellProcessing/scMatrix/AU_scMatrix/analysis_results/Auco.seurat.rds")
seurat_obj

In [ ]:
# 单细胞中的基因
assay_use <- DefaultAssay(seurat_obj)                 # e.g., "RNA" or "SCT"
seurat_genes <- SeuratObject::Features(seurat_obj, assay = assay_use) |> unique()
length(seurat_genes)

# eggNOG中的基因
eggNOG_genes <- eggNOG$gene |> unique()
length(eggNOG_genes)

# 背景基因：研究中被检测到的基因∩数据库能识别的基因
background_genes <- intersect(seurat_genes, eggNOG_genes)
length(background_genes)

ZZ：13绘制基因集合Venn图

In [ ]:
library(ggplot2)
library(ggVennDiagram)

# 两个集合
venn_list <- list(
  "snRNA genes" = seurat_genes,
  "eggNOG genes" = eggNOG_genes
)

p <- ggVennDiagram(
  venn_list,
  label_alpha = 0,        # 计数标签背景透明
  label = "count"         # 显示交/并集计数
) +
  theme_classic() +
  theme(
    # panel.border = element_rect(colour = "black", fill = NA, linewidth = 0.8),
    axis.line = element_blank(),
    axis.title = element_blank(),
    axis.text  = element_blank(),
    axis.ticks = element_blank(),
    legend.position = "none"
  )

# 保存 PDF（8×6）
pdf(paste0(fig_dir, "/13.Venn.seurat_vs_eggNOG.pdf"), width = 8, height = 8)
print(p)
dev.off()

# 保存 PNG（300 dpi，8×6）
ggsave(
  filename = paste0(fig_dir, "/13.Venn.seurat_vs_eggNOG.png"),
  plot = p, width = 8, height = 8, dpi = 300
)

p


In [ ]:
class(background_genes)
is.vector(background_genes)

### 4.The gene set of interest

In [ ]:
# 找每个cluster的top DEGs（默认方法：FindAllMarkers）
# 这里举例用Wilcox检验，筛选表达在至少10%细胞且logFC>0.25的基因
markers_all <- FindAllMarkers(
  seurat_obj,
  only.pos = FALSE,
  min.pct = 0,
  logfc.threshold = 0,
  test.use = "wilcox"
)

# 查看每个 cluster 的基因数
markers_all %>%
  group_by(cluster) %>%
  summarise(n_genes = n())

# 保存结果
write.csv(markers_all, "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/AllClusters_allGenes_Wilcox.csv", row.names = FALSE)


In [ ]:
cluster_id <- "7"  # 注意cluster列是字符还是数字，要对应
markers_table <- read_csv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/AllClusters_allGenes_Wilcox.csv")

interesting_genes <- markers_table %>%
  filter(cluster == cluster_id,
         abs(avg_log2FC) >= 1.5,
         p_val_adj <= 0.05) %>%
  dplyr::select(gene) %>%
  unlist() %>%
  as.vector()

length(interesting_genes)

### 5.ORA via clusterProfiler

In [ ]:
# perform ORA
term2gene <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/term2gene_GO.tsv")
term2name <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/term2name_GO.tsv")

enrichment <- enricher(interesting_genes,
                       TERM2GENE = term2gene,
                       TERM2NAME = term2name,
                       pvalueCutoff = 0.05,
                       universe = background_genes,
                       qvalueCutoff = 0.05)
#save the enrichment result
write.csv(file = paste0("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/enrichment_GO_results.csv"),                 # EDIT THIS
          x = enrichment@result)

if (any(enrichment@result$p.adjust <= 0.05)){
    p <- dotplot(enrichment,
                 x= "geneRatio", # Options: GeneRatio, BgRatio, pvalue, p.adjust, qvalue
                 color="p.adjust",
                 orderBy = "x", # Options: GeneRatio, BgRatio, pvalue, p.adjust, qvalue
                 showCategory=50,
                 font.size=8,
                 label_format = 200 # 标签属于长度
                 ) +
        ggtitle("dotplot for GO ORA")
    
    ggsave(filename = paste0("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/enrichment_GO_dotplot.pdf"),                # EDIT THIS
           plot =  p,  dpi = 300, width = 21, height = 42, units = "cm")
}

### 6.KEGG

In [ ]:
# === 1. 构建 term2gene / term2name ===
link <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/ko_pathway_link.txt", col_names = FALSE)
list <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/ko_pathway_list.txt", col_names = FALSE)

term2gene_kegg <- link %>%
  filter(grepl("^path:ko", X1)) %>%  # 只保留 path:ko 开头的行
  mutate(
    term = gsub("path:ko", "", X1),   # Pathway ID
    gene = gsub("ko:", "", X2)        # KO ID
  ) %>%
  select(term, gene) %>%
  distinct()

term2name_kegg <- list %>%
  mutate(
    term = gsub("ko", "", X1),        # 保证 term 与 term2gene 一致
    name = X2
  ) %>%
  select(term, name)

In [ ]:
head(term2gene_kegg)

In [ ]:
head(term2name_kegg)

In [ ]:
# === 2. 从 eggNOG 中提取 KEGG 对应 ===
eggNOG_kegg <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/eggNOG_output_processed/Auco.emapper.annotations.tsv") %>%
    dplyr::select(KEGG_ko, `query`) %>%
    dplyr::filter(KEGG_ko != "-") %>%
    separate_rows(KEGG_ko, sep = ",") %>%
    dplyr::mutate(gene = query) %>%
    dplyr::mutate(term = gsub("ko:", "", KEGG_ko)) %>%
    dplyr::select(term, gene) %>%
    distinct() %>%
    drop_na()
    
# === 3. 定义基因集合 ===
interesting_set_kegg <- eggNOG_kegg %>%
    dplyr::filter(gene %in% interesting_genes) %>%
    unlist() %>%
    as.vector()
# create a list of kegg ortholog that includes all kegg orthologs which form my background
background_kegg <- eggNOG_kegg %>%
    dplyr::filter(gene %in% background_genes) %>%
    unlist() %>%
    as.vector()

# enrichment_kegg <- enrichKEGG(interesting_set_kegg,
#            organism = "ko",
#            keyType = "kegg",
#            pvalueCutoff = 0.05,
#            pAdjustMethod = "BH",
#            universe = background_kegg,
#            minGSSize = 10,
#            maxGSSize = 500,
#            qvalueCutoff = 0.05,
#            use_internal_data = FALSE)

# === 4. 运行离线 KEGG 富集 ===
enrichment_kegg <- enricher(
  gene = interesting_set_kegg,        # 感兴趣基因（KO ID 或基因名）
  TERM2GENE = term2gene_kegg,      # pathway–gene 对照表
  TERM2NAME = term2name_kegg,      # pathway 名称表
  universe = background_kegg,     # 背景基因
  pAdjustMethod = "BH",
  pvalueCutoff = 0.05,
  qvalueCutoff = 0.05
)
#save the enrichment result
write.csv(file = paste0("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/enrichment_KEGG_results.csv"),                 # EDIT THIS
          x = enrichment_kegg@result)

if (any(enrichment_kegg@result$p.adjust <= 0.05)){
    p <- dotplot(enrichment_kegg,
                 x= "geneRatio", # Options: GeneRatio, BgRatio, pvalue, p.adjust, qvalue
                 color="p.adjust",
                 orderBy = "x", # Options: GeneRatio, BgRatio, pvalue, p.adjust, qvalue
                 showCategory=100,
                 font.size=8) +
        ggtitle("dotplot for KEGG ORA")
    
    ggsave(filename = paste0("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/enrichment_KEGG_dotplot.pdf"),                # EDIT THIS
           plot =  p,  dpi = 300, width = 21, height = 42, units = "cm")
}


### 7.GSEA

In [ ]:
# GSEA
# This time I am not filtering the changes
# gsea_list <- read_tsv("comparison_6.tsv") %>%
#     dplyr::arrange(desc(logFC))
gsea_list <- markers_table %>%
    filter(
        cluster == cluster_id
    ) %>%
    dplyr::arrange(desc(avg_log2FC))
gsea_input <- gsea_list %>%
    # dplyr::select(logFC) %>%
    dplyr::select(avg_log2FC) %>%
    unlist() %>%
    as.vector()
names(gsea_input) <- gsea_list$gene
head(gsea_input)

In [ ]:

# do the analysis below
enrichment_gsea <- GSEA(geneList = gsea_input,
                        TERM2GENE = term2gene,
                        TERM2NAME = term2name,
                        minGSSize = 10,
                        maxGSSize = 500,
                        eps = 1e-10,
                        pvalueCutoff = 0.05,
                        pAdjustMethod = "BH")
#save the enrichment result
write.csv(file = paste0("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/enrichment_GSEA_results.csv"),                 # EDIT THIS
          x = enrichment_gsea@result)

if (any(enrichment_gsea@result$p.adjust <= 0.05)){
    # p <- ridgeplot(enrichment_gsea,
    #                core_enrichment= FALSE,
    #                fill="p.adjust",
    #                orderBy = "NES",
    #                showCategory=100) +
    #     ggtitle("Ridge plot for GSEA")
    p <- dotplot(enrichment_gsea,
             showCategory = 30,
             split = NULL,
             color = "p.adjust",
             label_format = 200) +
     ggtitle(paste0("GSEA dotplot (cluster ", cluster_id, ")")) +
     theme(plot.title = element_text(size = 14, face = "bold"))
    
    ggsave(filename = paste0("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/enrichment_GSEA_dotplot.pdf"),                # EDIT THIS
           plot =  p,  dpi = 300, width = 21, height = 42, units = "cm")
}

ZZ：14绘制cluster7的GSEA图

In [ ]:

# do the analysis below
enrichment_gsea <- GSEA(geneList = gsea_input,
                        TERM2GENE = term2gene,
                        TERM2NAME = term2name,
                        minGSSize = 10,
                        maxGSSize = 500,
                        eps = 1e-10,
                        pvalueCutoff = 0.05,
                        pAdjustMethod = "BH")
#save the enrichment result
write.csv(file = paste0("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/enrichment_GSEA_results.csv"),                 # EDIT THIS
          x = enrichment_gsea@result)

if (any(enrichment_gsea@result$p.adjust <= 0.05)){
    # p <- ridgeplot(enrichment_gsea,
    #                core_enrichment= FALSE,
    #                fill="p.adjust",
    #                orderBy = "NES",
    #                showCategory=100) +
    #     ggtitle("Ridge plot for GSEA")
    p <- dotplot(enrichment_gsea,
             showCategory = 30,
             split = NULL,
             color = "p.adjust",
             label_format = 200) +
     ggtitle(paste0("GSEA dotplot (cluster ", cluster_id, ")")) +
     theme(plot.title = element_text(size = 14, face = "bold"))
    
    ggsave(
        filename = paste0(fig_dir, "/14.DotPlot.cluster", cluster_id, "_GSEA.pdf"),
        plot = p, width = 8, height = 8, dpi = 300
        # plot =  p,  dpi = 300, width = 21, height = 42, units = "cm"
    )
}

### 8.GO多个cluster的自动分析

In [ ]:
save_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/cluster_GO_results/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# 提取所有 cluster ID
all_clusters <- sort(unique(markers_table$cluster))
cat("Detected clusters:", all_clusters, "\n")

for (cluster_id in all_clusters) {
    interesting_genes <- markers_table %>%
    filter(cluster == cluster_id,
            abs(avg_log2FC) >= 1.5,
            p_val_adj <= 0.05) %>%
    dplyr::select(gene) %>%
    unlist() %>%
    as.vector()

    length(interesting_genes)

    # perform ORA
    term2gene <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/term2gene_GO.tsv")
    term2name <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/term2name_GO.tsv")

    enrichment <- enricher(interesting_genes,
                        TERM2GENE = term2gene,
                        TERM2NAME = term2name,
                        pvalueCutoff = 0.05,
                        universe = background_genes,
                        qvalueCutoff = 0.05)
    #save the enrichment result
    write.csv(file = paste0(save_dir, "cluster_", cluster_id, "_GO_results.csv"),                 # EDIT THIS
            x = enrichment@result)

    if (any(enrichment@result$p.adjust <= 0.05)){
        p <- dotplot(enrichment,
                    x= "geneRatio", # Options: GeneRatio, BgRatio, pvalue, p.adjust, qvalue
                    color="p.adjust",
                    orderBy = "x", # Options: GeneRatio, BgRatio, pvalue, p.adjust, qvalue
                    showCategory=50,
                    font.size=8,
                    label_format = 200 # 标签属于长度
                    ) +
            ggtitle("dotplot for GO ORA")
        
        ggsave(filename = paste0(save_dir, "cluster_", cluster_id, "_GO_dotplot.pdf"),                # EDIT THIS
            plot =  p,  dpi = 300, width = 21, height = 42, units = "cm")
    }
}

### 9.KEGG多个cluster的自动分析

In [ ]:
# === 1. 构建 term2gene / term2name ===
link <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/ko_pathway_link.txt", col_names = FALSE)
list <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/ko_pathway_list.txt", col_names = FALSE)

term2gene_kegg <- link %>%
  filter(grepl("^path:ko", X1)) %>%  # 只保留 path:ko 开头的行
  mutate(
    term = gsub("path:ko", "", X1),   # Pathway ID
    gene = gsub("ko:", "", X2)        # KO ID
  ) %>%
  select(term, gene) %>%
  distinct()

term2name_kegg <- list %>%
  mutate(
    term = gsub("ko", "", X1),        # 保证 term 与 term2gene 一致
    name = X2
  ) %>%
  select(term, name)

In [ ]:
# === 2. 从 eggNOG 中提取 KEGG 对应 ===
eggNOG_kegg <- read_tsv("/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/eggNOG_output_processed/Auco.emapper.annotations.tsv") %>%
    dplyr::select(KEGG_ko, `query`) %>%
    dplyr::filter(KEGG_ko != "-") %>%
    separate_rows(KEGG_ko, sep = ",") %>%
    dplyr::mutate(gene = query) %>%
    dplyr::mutate(term = gsub("ko:", "", KEGG_ko)) %>%
    dplyr::select(term, gene) %>%
    distinct() %>%
    drop_na()

In [ ]:
save_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/cluster_KEGG_results/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# 提取所有 cluster ID
all_clusters <- sort(unique(markers_table$cluster))
cat("Detected clusters:", all_clusters, "\n")

for (cluster_id in all_clusters) {
    # === 3. 定义基因集合 ===
    interesting_genes <- markers_table %>%
    filter(cluster == cluster_id,
            abs(avg_log2FC) >= 1.5,
            p_val_adj <= 0.05) %>%
    dplyr::select(gene) %>%
    unlist() %>%
    as.vector()

    length(interesting_genes)

    interesting_set_kegg <- eggNOG_kegg %>%
        dplyr::filter(gene %in% interesting_genes) %>%
        unlist() %>%
        as.vector()
    # create a list of kegg ortholog that includes all kegg orthologs which form my background
    background_kegg <- eggNOG_kegg %>%
        dplyr::filter(gene %in% background_genes) %>%
        unlist() %>%
        as.vector()

    # === 4. 运行离线 KEGG 富集 ===
    enrichment_kegg <- enricher(
    gene = interesting_set_kegg,        # 感兴趣基因（KO ID 或基因名）
    TERM2GENE = term2gene_kegg,      # pathway–gene 对照表
    TERM2NAME = term2name_kegg,      # pathway 名称表
    universe = background_kegg,     # 背景基因
    pAdjustMethod = "BH",
    pvalueCutoff = 0.05,
    qvalueCutoff = 0.05
    )
    #save the enrichment result
    write.csv(file = paste0(save_dir, "cluster_", cluster_id, "_KEGG_results.csv"),                 # EDIT THIS
            x = enrichment_kegg@result)

    if (any(enrichment_kegg@result$p.adjust <= 0.05)){
        p <- dotplot(enrichment_kegg,
                    x= "geneRatio", # Options: GeneRatio, BgRatio, pvalue, p.adjust, qvalue
                    color="p.adjust",
                    orderBy = "x", # Options: GeneRatio, BgRatio, pvalue, p.adjust, qvalue
                    showCategory=100,
                    font.size=8) +
            ggtitle("dotplot for KEGG ORA")
        
        ggsave(filename = paste0(save_dir, "cluster_", cluster_id, "_KEGG_dotplot.pdf"),                # EDIT THIS
            plot =  p,  dpi = 300, width = 21, height = 42, units = "cm")
    }
}

### 10.GSEA多个cluster的自动分析

In [ ]:
save_dir <- "/share/home/zhangze/zz/NeuralOrigin/Data/06.SingleCellAnalysis/eggNOG_ClusterProfiler_GO/cluster_GSEA_results/"
dir.create(save_dir, showWarnings = FALSE, recursive = TRUE)

# 提取所有 cluster ID
all_clusters <- sort(unique(markers_table$cluster))
cat("Detected clusters:", all_clusters, "\n")

for (cluster_id in all_clusters) {
    gsea_list <- markers_table %>%
        filter(
            cluster == cluster_id
        ) %>%
        dplyr::arrange(desc(avg_log2FC))
    gsea_input <- gsea_list %>%
        # dplyr::select(logFC) %>%
        dplyr::select(avg_log2FC) %>%
        unlist() %>%
        as.vector()
    names(gsea_input) <- gsea_list$gene
    head(gsea_input)

    # do the analysis below
    enrichment_gsea <- GSEA(geneList = gsea_input,
                            TERM2GENE = term2gene,
                            TERM2NAME = term2name,
                            minGSSize = 10,
                            maxGSSize = 500,
                            eps = 1e-10,
                            pvalueCutoff = 0.05,
                            pAdjustMethod = "BH")
    #save the enrichment result
    write.csv(file = paste0(save_dir, "cluster_", cluster_id, "_GSEA_results.csv"),                 # EDIT THIS
            x = enrichment_gsea@result)

    if (any(enrichment_gsea@result$p.adjust <= 0.05)){
        # p <- ridgeplot(enrichment_gsea,
        #                core_enrichment= FALSE,
        #                fill="p.adjust",
        #                orderBy = "NES",
        #                showCategory=100) +
        #     ggtitle("Ridge plot for GSEA")
        p <- dotplot(enrichment_gsea,
                showCategory = 30,
                split = NULL,
                color = "p.adjust",
                label_format = 200) +
        ggtitle(paste0("GSEA dotplot (cluster ", cluster_id, ")")) +
        theme(plot.title = element_text(size = 14, face = "bold"))
        
        ggsave(filename = paste0(save_dir, "cluster_", cluster_id, "_GSEA_dotplot.pdf"),                # EDIT THIS
            plot =  p,  dpi = 300, width = 21, height = 42, units = "cm")
    }
}

### 11.GSEA 后续分析

In [ ]:
library(enrichplot)

In [ ]:
gseaplot2(
    enrichment_gsea,
    geneSetID = "GO:0022843",
    title = "GO:0022843",
    base_size = 10,
    rel_heights = c(1, 0.2, 0.4),#小图相对高度
    subplots = 1:3,#展示小图
    pvalue_table = TRUE,#p值表格
    ES_geom = "line"#line or dot
  )

ZZ：16绘制cluster7的GSEA曲线分析

In [ ]:
library(enrichplot)
library(ggplot2)

pdf(paste0(fig_dir, "/16.Curve.GSEA_GO_0022843.pdf"), width = 12, height = 8)

print(
  gseaplot2(
      enrichment_gsea,
      geneSetID = "GO:0022843",
      title = "GO:0022843",
      base_size = 15,
      rel_heights = c(1, 0.2, 0.4),#小图相对高度
      subplots = 1:3,#展示小图
      pvalue_table = TRUE,#p值表格
      ES_geom = "line"#line or dot
    )
)

dev.off()